# Model Training: Student Performance Predictor

This notebook mirrors the production training script while keeping the workflow inspectable: preprocessing, feature engineering, model comparison, hyperparameter tuning, evaluation, and artifact saving.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.feature_engineering import get_model_feature_columns
from src.preprocessing import split_dataset
from src.train import (
    classification_models,
    evaluate_classification,
    evaluate_regression,
    prepare_dataset,
    regression_models,
    train_baselines,
    tune_classifier_model,
    tune_regression_model,
)
from src.utils import MODELS_DIR, save_joblib

## Prepare Data

The training dataset is cleaned, deduplicated, range-corrected, and enriched with engineered features before the train/test split.

In [ ]:
data = prepare_dataset()
features = get_model_feature_columns(data)
split = split_dataset(data, features)
data.head()

## Regression Model Comparison

The regression task predicts the final score. Lower MAE/RMSE and higher R2 indicate better score prediction.

In [ ]:
regression_table, regression_fitted = train_baselines(
    regression_models(),
    split.x_train,
    split.y_reg_train,
    split.x_test,
    split.y_reg_test,
    task='regression',
)
regression_table.sort_values('RMSE')

## Classification Model Comparison

The classification task predicts pass/fail. F1 is emphasized because it balances precision and recall for the pass/fail decision.

In [ ]:
classification_table, classification_fitted = train_baselines(
    classification_models(),
    split.x_train,
    split.y_clf_train,
    split.x_test,
    split.y_clf_test,
    task='classification',
)
classification_table.sort_values('F1', ascending=False)

## Hyperparameter Tuning

RandomizedSearchCV explores Random Forest capacity, splitting rules, and feature sampling. The searches are compact enough for a portfolio laptop while still demonstrating proper model selection.

In [ ]:
regression_search = tune_regression_model(split.x_train, split.y_reg_train)
classifier_search = tune_classifier_model(split.x_train, split.y_clf_train)

best_regression = regression_search.best_estimator_
best_classifier = classifier_search.best_estimator_

print('Best regression params:', regression_search.best_params_)
print('Best classifier params:', classifier_search.best_params_)

## Final Evaluation

The tuned models are evaluated on the untouched test split to estimate generalization performance.

In [ ]:
final_regression_metrics = evaluate_regression(best_regression, split.x_test, split.y_reg_test)
final_classification_metrics = evaluate_classification(best_classifier, split.x_test, split.y_clf_test)

display(pd.DataFrame([final_regression_metrics], index=['Tuned Random Forest Regressor']))
display(pd.DataFrame([final_classification_metrics], index=['Tuned Random Forest Classifier']))

## Save Models

The final artifacts are saved as complete sklearn pipelines. Each pipeline includes preprocessing, scaling, categorical encoding, and the trained estimator.

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
save_joblib(best_regression, MODELS_DIR / 'regression.pkl')
save_joblib(best_classifier, MODELS_DIR / 'classifier.pkl')
print(f'Saved models to {MODELS_DIR}')